# RumourEval-2017 Stance Detection — Twitter-RoBERTa

Fine-tunes `cardiffnlp/twitter-roberta-base` on the PHEME dataset using
Leave-One-Event-Out (LOEO) cross-validation.

**Before running:** make sure the runtime is set to **GPU** (T4 or better).
Runtime → Change runtime type → T4 GPU.

## 1 — Verify GPU

In [ ]:
!nvidia-smi

## 2 — Clone repo

In [ ]:
import os

REPO_URL  = "https://github.com/davisclrk/RoBERTa-stance-analysis"
REPO_NAME = "RoBERTa-stance-analysis"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.isdir(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}
else:
    print("Repo already cloned — pulling latest changes.")
    !git -C {REPO_PATH} pull

%cd {REPO_PATH}

## 3 — Download & extract PHEME dataset

Download the PHEME rumour scheme dataset (~50 MB) from figshare and upload the tar.gz to colab files `data/raw/`, then run this cell to extract it.

In [14]:
!bash scripts/extract_data.sh

Extracting...
Done. Dataset available at data/raw/pheme-rumour-scheme-dataset


## 4 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 5 — Sanity-check the data pipeline

In [16]:
!ls -la data/raw/
!ls data/raw/pheme-rumour-scheme-dataset/

total 66520
drwxr-xr-x 3 root root     4096 Apr 30 04:54 .
drwxr-xr-x 4 root root     4096 Apr 30 04:21 ..
-rw-r--r-- 1 root root        0 Apr 30 04:21 .gitkeep
drwxrwxr-x 4 1000 1000     4096 Apr 21  2016 pheme-rumour-scheme-dataset
-rw-r--r-- 1 root root 68100705 Apr 30 04:53 phemerumourschemedataset.tar.bz2
annotations  annotation-scheme.json  README  threads


In [15]:
import sys
sys.path.insert(0, "src")

from collections import Counter
from data import load_pheme_dataset, loeo_splits

examples = load_pheme_dataset()
label_names = ["support", "deny", "query", "comment"]
counts = Counter(ex["label"] for ex in examples)

print(f"Total examples : {len(examples)}")
print(f"Label distribution:")
for i, name in enumerate(label_names):
    print(f"  {name:<10} {counts[i]:>4}  ({100*counts[i]/len(examples):.1f}%)")

splits = loeo_splits(examples)
print(f"\nLOEO folds:")
for event, train, test in splits:
    print(f"  held-out={event:<22}  train={len(train):>4}  test={len(test):>4}")

Total examples : 4263
Label distribution:
  support     645  (15.1%)
  deny        334  (7.8%)
  query       361  (8.5%)
  comment    2923  (68.6%)

LOEO folds:
  held-out=putinmissing            train=4205  test=  58
  held-out=charliehebdo            train=3261  test=1002
  held-out=prince-toronto          train=4172  test=  91
  held-out=ferguson                train=3221  test=1042
  held-out=germanwings-crash       train=4002  test= 261
  held-out=ottawashooting          train=3539  test= 724
  held-out=sydneysiege             train=3210  test=1053
  held-out=ebola-essien            train=4231  test=  32


## 6 — Train (LOEO)

Runs all 8 Leave-One-Event-Out folds, each with `NUM_SEEDS` independent seeds (default 3). Each fold/seed combination:
- Loads `twitter-roberta-base` and adds a 4-way classification head
- Uses layer-wise LR decay (`LR_DECAY=0.9`) so lower encoder layers update more slowly
- Uses sqrt-inverse-frequency class-weighted CE loss to handle the ~9:1 comment/deny imbalance (no `WeightedRandomSampler` — combining the two over-corrected and starved the comment class)
- Trains up to `NUM_EPOCHS=10` epochs, with early stopping after `EARLY_STOP_PATIENCE=2` epochs of no macro-F1 improvement
- Saves the best checkpoint (by macro-F1) to `outputs/fold_{event}/seed_{N}/best_model.pt`

Per-fold and overall results are aggregated to mean ± std across seeds.

**Expected runtime:** ~60–90 min on a T4 GPU for all 8 folds × 3 seeds (early stopping caps it well below the 8 × 3 × 10 = 240-epoch worst case). Reduce `NUM_SEEDS` to 1 in `src/config.py` for a quick smoke run.

In [ ]:
from train import run_loeo

results = run_loeo()

Device: cuda  |  seeds per fold: 3

Fold: held-out=putinmissing | train=4205  test=58


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  [charliehebdo seed=44] epoch 3  loss=1.085  macro_f1=0.4959  acc=0.6527


  [charliehebdo seed=44] epoch 4  loss=0.992  macro_f1=0.4538  acc=0.5649


  [charliehebdo seed=44] epoch 5  loss=0.924  macro_f1=0.4738  acc=0.6098
  [charliehebdo seed=44] early stop at epoch 5 (best epoch 3, macro_f1=0.4959)
              precision    recall  f1-score   support

     support       0.47      0.58      0.52       172
        deny       0.10      0.19      0.13        54
       query       0.47      0.75      0.58        53
     comment       0.83      0.70      0.76       723

    accuracy                           0.65      1002
   macro avg       0.47      0.55      0.50      1002
weighted avg       0.71      0.65      0.67      1002

  [charliehebdo] across 3 seeds: macro_f1=0.5047±0.0064  acc=0.6816±0.0205

Fold: held-out=prince-toronto | train=4172  test=91


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[prince-toronto seed=42] epoch 1/10:  17%|█▋        | 44/261 [00:09<00:46,  4.72it/s, loss=1.417]

## 7 — Inspect results

In [ ]:
import json
import numpy as np
from pathlib import Path
from IPython.display import Image, display

# Load saved results if re-running this cell after training
results_path = Path("outputs/loeo_results.json")
if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)

macro_means = [v["macro_f1_mean"] for v in results.values()]
print(f"Mean macro-F1 across all folds: {np.mean(macro_means):.4f}")
print()

for event, m in results.items():
    print(
        f"{event:<26}  "
        f"macro_f1={m['macro_f1_mean']:.4f}±{m['macro_f1_std']:.4f}  "
        f"acc={m['accuracy_mean']:.4f}±{m['accuracy_std']:.4f}"
    )
    for name, f1_mean, f1_std in zip(label_names, m["per_class_f1_mean"], m["per_class_f1_std"]):
        print(f"    {name:<10} F1={f1_mean:.4f}±{f1_std:.4f}")
    print()

In [ ]:
# Show the seed_0 confusion matrix for each fold (representative — per-seed
# matrices are saved under outputs/fold_{event}/seed_{seed}/).
for event in results:
    fold_dir = Path(f"outputs/fold_{event}")
    seed_dirs = sorted(fold_dir.glob("seed_*"))
    if not seed_dirs:
        continue
    img_path = seed_dirs[0] / "confusion_matrix.png"
    if img_path.exists():
        print(f"\n--- {event} (seed {seed_dirs[0].name}) ---")
        display(Image(str(img_path)))

## 8 — Download outputs

Zips all checkpoints and results and downloads them to your local machine.

In [ ]:
!zip -r /content/outputs.zip outputs/
from google.colab import files
files.download("/content/outputs.zip")